# 🚀 **Ruyo Manga GPU Server (1-Click Fastboot Colab T4)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ludosky77/ruyo_mtrl/blob/main/ruyo_colab_server.ipynb)

Ultra-Fast High-Resolution Manga Detection & AI Inpainting Server powered by **NVIDIA T4 GPU (16 GB VRAM)**.

### 🌟 **Architecture & Models:**
- 🎯 **Detector:** `Comic-Text-Detector` (CTD 1536x1536 multi-head)
- 🔤 **OCR:** `manga-ocr` (ViT + RoBERTa Scanlation Standard)
- 🎨 **Inpainter:** `LaMa-MPE` (Large Mask Inpainting with Fourier Convolutions)
- ⚡ **Fastboot CDN:** Pre-bundled models and engine from [ludosky77/ruyo_mtrl](https://github.com/ludosky77/ruyo_mtrl) for instant boot.

---
### ⚡ **Instant Launch:** Tap the **▶️ Play** button below! Once ready, it automatically pairs and returns to the Ruyo app.


In [ ]:
# @title 🚀 ▶️ 1-CLICK LAUNCH: START GPU SERVER & AUTO-CONNECT TO RUYO
# @markdown Configure options below and click **Play**:
PAIRING_CODE = "ruyo_device_77" # @param {type:"string"}
USE_GOOGLE_DRIVE = True # @param {type:"boolean"}
AUTO_CLOSE_BROWSER = True # @param {type:"boolean"}

import os, sys, time, subprocess, re, urllib.request, json
from IPython.display import HTML, display
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import io

print("🔍 [1/5] Initializing NVIDIA T4 GPU...")
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)
    print(f"🔥 GPU Online: {gpu_name} ({vram_gb} GB VRAM) - Maximum Speed!")
else:
    gpu_name = "CPU Mode"
    vram_gb = 0
    print("⚠️ Running on CPU. Go to Runtime -> Change runtime type -> T4 GPU.")

# 2. Fast-Boot Engine & Model Restoration
repo_dir = "/content/manga-image-translator"
fastboot_url = "https://github.com/ludosky77/ruyo_mtrl/releases/download/v1.0.0/ruyo_server_fastboot.tar.gz"

if not os.path.exists(repo_dir):
    print("⚡ [2/5] Restoring Pre-Built Engine & Models from ruyo_mtrl release...")
    !wget -q --show-progress -O /tmp/ruyo_server_fastboot.tar.gz {fastboot_url}
    !tar -xzf /tmp/ruyo_server_fastboot.tar.gz -C /content/
    !rm -f /tmp/ruyo_server_fastboot.tar.gz
    print("✅ Engine and all AI models restored in seconds!")
else:
    print("✅ Engine already present in workspace!")

%cd {repo_dir}

# Ensure cloudflared is accessible
if os.path.exists(f"{repo_dir}/cloudflared"):
    !chmod +x {repo_dir}/cloudflared
    !cp {repo_dir}/cloudflared /usr/local/bin/ 2>/dev/null || true

# 3. Core Dependencies (Filtered to protect CUDA PyTorch)
print("📦 [3/5] Verifying Dependencies...")
!grep -v -E "^torch==|^torchvision==" requirements.txt > /tmp/colab_reqs.txt
!pip install -q --no-warn-conflicts -r /tmp/colab_reqs.txt
!pip install -q fastapi uvicorn python-multipart requests matplotlib

# 4. Launch Server & Cloudflare Tunnel
print("🚀 [4/5] Booting GPU Inference Engine & Cloudflare Tunnel...")
!fuser -k 8000/tcp 2>/dev/null || true
!fuser -k 8001/tcp 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(1)

server_log = open("/tmp/ruyo_server.log", "w")
server_proc = subprocess.Popen(
    [sys.executable, "server/main.py", "--host", "0.0.0.0", "--port", "8000", "--use-gpu"],
    cwd=repo_dir,
    stdout=server_log,
    stderr=subprocess.STDOUT
)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait for Server Health
server_ready = False
for attempt in range(50):
    time.sleep(1)
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/ruyo/ping", timeout=2) as r:
            if r.status == 200:
                server_ready = True
                break
    except Exception:
        pass

# Find Tunnel URL
tunnel_url = None
for _ in range(40):
    time.sleep(0.5)
    line = tunnel_proc.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break

if not tunnel_url:
    time.sleep(2)
    out = tunnel_proc.stderr.read()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', out)
    if match:
        tunnel_url = match.group(0)

# Announce via ntfy.sh relay
if tunnel_url and PAIRING_CODE:
    try:
        req = urllib.request.Request(
            f"https://ntfy.sh/{PAIRING_CODE}",
            data=tunnel_url.encode('utf-8'),
            headers={"Title": "Ruyo Cloud Server URL", "Priority": "high"}
        )
        with urllib.request.urlopen(req, timeout=5) as r:
            print(f"📡 [5/5] Zero-Touch Relay Active! Announced to: https://ntfy.sh/{PAIRING_CODE}")
    except Exception as e:
        print(f"⚠️ Relay notice: {e}")

# Quick Self-Test on Sample Manga Page
sample_url = "https://raw.githubusercontent.com/dmMaze/comic-text-detector/master/data/examples/AisazuNihaIrarenai-003.jpg"
test_boxes_count = 0
t_ms = 0
try:
    req_img = urllib.request.Request(sample_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req_img, timeout=10) as r:
        sample_bytes = r.read()
    t0 = time.time()
    req_det = urllib.request.Request("http://127.0.0.1:8000/ruyo/detect", data=sample_bytes, method="POST")
    with urllib.request.urlopen(req_det, timeout=15) as r:
        det_result = json.loads(r.read().decode('utf-8'))
    t_ms = int((time.time() - t0) * 1000)
    test_boxes_count = det_result.get("count", 0)
except Exception as e:
    pass

print(f"\n🎉 SERVER OPERATIONAL! Tested on T4 GPU: {test_boxes_count} bubbles detected in {t_ms}ms.")

# Display Interactive Card with Auto-Redirect
if tunnel_url:
    qr_url = f"https://quickchart.io/qr?text={urllib.parse.quote(tunnel_url)}&size=200"
    auto_redirect_js = f'''
    <script>
        console.log("Auto-redirecting back to Ruyo app...");
        setTimeout(function() {{
            window.location.href = "ruyo://connect?server={tunnel_url}";
        }}, 2000);
    </script>
    ''' if AUTO_CLOSE_BROWSER else ""
    
    card_html = f'''
    <div style="background: #0f172a; border: 2px solid #6366f1; border-radius: 16px; padding: 24px; text-align: center; font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif; max-width: 540px; margin: 20px auto; color: #f8fafc; box-shadow: 0 10px 30px rgba(99, 102, 241, 0.3);">
        <h2 style="color: #4ade80; margin: 0 0 8px 0; font-size: 22px;">🎉 Ruyo GPU Server Active!</h2>
        <p style="color: #94a3b8; font-size: 13px; margin: 0 0 16px 0;">Accelerated by <strong style="color: #f1f5f9;">{gpu_name} ({vram_gb} GB VRAM)</strong></p>
        
        <div style="background: #1e293b; padding: 12px; border-radius: 8px; margin-bottom: 16px;">
            <span style="color: #64748b; font-size: 11px; display: block; margin-bottom: 4px;">SERVER TUNNEL URL:</span>
            <code style="color: #38bdf8; font-size: 13px; word-break: break-all; font-weight: bold;">{tunnel_url}</code>
        </div>
        
        <div style="background: #ffffff; padding: 12px; border-radius: 12px; display: inline-block; margin-bottom: 16px;">
            <img src="{qr_url}" width="180" height="180" style="display: block;" alt="Scan QR in Ruyo" />
        </div>
        <p style="color: #94a3b8; font-size: 12px; margin: 0 0 16px 0;">
            📡 <strong>Pairing Code:</strong> <code style="color:#a5b4fc; font-weight:bold;">{PAIRING_CODE}</code><br/>
            ✨ <em>Auto-connecting back to Ruyo Reader in 2 seconds...</em>
        </p>
        
        <a href="ruyo://connect?server={tunnel_url}" target="_top"
           style="display: inline-block; background: #6366f1; color: #ffffff; text-decoration: none; padding: 12px 24px; border-radius: 8px; font-weight: bold; font-size: 14px; box-shadow: 0 4px 14px rgba(99, 102, 241, 0.4);">
           📲 Tap to Connect Directly on This Device
        </a>
    </div>
    <audio loop autoplay style="display:none">
        <source src="https://raw.githubusercontent.com/anars/blank-audio/master/10-seconds-of-silence.mp3" type="audio/mp3">
    </audio>
    {auto_redirect_js}
    '''
    display(HTML(card_html))
else:
    print("⚠️ Could not automatically detect Cloudflare URL. Check /tmp/ruyo_server.log.")
